In [1]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "ngsolve"
solver2 = "moose"

sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}.vtu")
sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")

In [2]:
tree = KDTree(sol_ngsolve.points)
distances, indices = tree.query(sol_moose.points)

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")

aligned_grid = sol_moose.copy()


for array_name in sol_ngsolve.point_data.keys():
        data = sol_ngsolve.point_data[array_name]
        
        reordered_data = data[indices]
        
        aligned_grid.point_data[array_name] = reordered_data
        print(f"Transferred array: {array_name}")

aligned_grid.save(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")

Maximum alignment error (distance): 8.396113e-11
Transferred array: magnetic_vector_potential_nd
Transferred array: magnetic_flux_density_nd
Transferred array: current_density
Transferred array: magnetic_vector_potential
Transferred array: electric_potential
Transferred array: magnetic_flux_density


In [3]:
sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")
# elec_pot_ngsolve = sol_ngsolve["electric_potential"]
mag_flux_ngsolve = sol_ngsolve["magnetic_flux_density"]
mag_vec_ngsolve = sol_ngsolve["magnetic_vector_potential"]

sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")
# elec_pot_moose = sol_moose["electric_potential"]
mag_flux_moose = sol_moose["magnetic_flux_density"]
mag_vec_moose = sol_moose["magnetic_vector_potential"]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_moose.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_moose.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_moose.shape)

(278516, 3)
(278516, 3)
(278516, 3)
(278516, 3)


In [4]:
mesh = sol_moose.copy()

mesh.point_data.remove("magnetic_vector_potential")
mesh.point_data.remove("magnetic_flux_density")

In [5]:
# print(f"Electric potential errors between {solver1} and {solver2}:")

# mesh = error(sol=elec_pot_ngsolve, sol_ref=elec_pot_moose, 
#              eps = 1e-6, mesh=mesh, tag="scalar", save_tag="V")

In [6]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_moose, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="B")

Magnetic flux density errors between ngsolve and moose:

  * Max. absolute error in x direction  : 6.699e+04.
  * Avg. absolute error in x direction  : 5.856e+03.

  * Max. relative error in x direction : 8.796e+04 %.
  * Avg. relative error in x direction : 1.717e+02 %.

  * Max. absolute error in y direction  : 6.790e+04.
  * Avg. absolute error in y direction  : 4.148e+03.

  * Max. relative error in y direction : 2.218e+05 %.
  * Avg. relative error in y direction : 2.810e+02 %.

  * Max. absolute error in z direction  : 8.599e+04.
  * Avg. absolute error in z direction  : 7.988e+03.

  * Max. relative error in z direction : 1.866e+05 %.
  * Avg. relative error in z direction : 1.770e+02 %.


/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: divide by zero encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)


In [7]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_moose, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="A")

Magnetic vector potential errors between ngsolve and moose:

  * Max. absolute error in x direction  : 1.706e+14.
  * Avg. absolute error in x direction  : 5.432e+11.

  * Max. relative error in x direction : 6.801e+17 %.
  * Avg. relative error in x direction : 4.073e+13 %.

  * Max. absolute error in y direction  : 1.188e+14.
  * Avg. absolute error in y direction  : 6.007e+11.

  * Max. relative error in y direction : 7.994e+06 %.
  * Avg. relative error in y direction : 1.907e+03 %.

  * Max. absolute error in z direction  : 1.436e+14.
  * Avg. absolute error in z direction  : 4.230e+11.

  * Max. relative error in z direction : 6.123e+17 %.
  * Avg. relative error in z direction : 2.202e+13 %.


In [8]:
mesh.save(f"../../output/{case_name}/{case_name}_error_ngsolve_moose.vtu")